In [1]:
pip install scikit-surprise

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np

from surprise import Dataset
from surprise import Reader
from surprise import SVD

from surprise.model_selection import train_test_split
from surprise import accuracy

import pickle

print("Libraries Loaded Successfully")

Libraries Loaded Successfully


In [3]:
recommendation_df = pd.read_csv(
    r"E:\Reel-Recommendation-Engine\data\processed\recommendation_dataset.csv"
)

svd_df = recommendation_df[
    [
        "user_id",
        "video_id",
        "recommendation_score"
    ]
].copy()

print(svd_df.shape)

svd_df.head()

(1414622, 3)


,user_id,video_id,recommendation_score
0,0,1527,0.058969
1,0,7405,0.007104
2,0,6026,0.052402
3,1,6354,0.009077
4,1,3645,0.018201


In [4]:
svd_df["recommendation_score"].describe()

count    1.409646e+06
mean     8.754370e-02
std      1.386591e-01
min      0.000000e+00
25%      1.471320e-02
50%      3.793040e-02
75%      9.398604e-02
max      1.000000e+00
Name: recommendation_score, dtype: float64

In [5]:
svd_df = svd_df.dropna(
    subset=["recommendation_score"]
)

print(svd_df.shape)

print(
    "Remaining NaN:",
    svd_df["recommendation_score"].isnull().sum()
)

(1409646, 3)
Remaining NaN: 0


In [6]:
reader = Reader(
    rating_scale=(0, 1)
)

data = Dataset.load_from_df(
    svd_df,
    reader
)

print("Dataset Created")

Dataset Created


In [7]:
trainset, testset = train_test_split(
    data,
    test_size=0.2,
    random_state=42
)

print("Train/Test Split Completed")

Train/Test Split Completed


In [8]:
svd_model = SVD(
    n_factors=100,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42
)

svd_model.fit(trainset)

print("SVD Training Completed")

SVD Training Completed


In [9]:
predictions = svd_model.test(
    testset
)

rmse = accuracy.rmse(
    predictions
)

print("RMSE:", rmse)

RMSE: 0.0316
RMSE: 0.03163412979640233


In [10]:
sample_user = svd_df["user_id"].iloc[0]

sample_video = svd_df["video_id"].iloc[0]

prediction = svd_model.predict(
    sample_user,
    sample_video
)

print(prediction.est)

0.06867298655409385


In [11]:
all_videos = svd_df["video_id"].unique()

In [12]:
def recommend_for_user(
        user_id,
        top_n=10):

    predictions = []

    for video_id in all_videos:

        pred = svd_model.predict(
            user_id,
            video_id
        )

        predictions.append(
            (
                video_id,
                pred.est
            )
        )

    predictions = sorted(
        predictions,
        key=lambda x: x[1],
        reverse=True
    )

    return predictions[:top_n]

In [13]:
sample_user = svd_df["user_id"].iloc[0]

recommend_for_user(sample_user)

[(1868, 0.9640118620550967),
 (3310, 0.6384293038561041),
 (2399, 0.6300285905904728),
 (2303, 0.6210722047389368),
 (6921, 0.5870048796941305),
 (7136, 0.4991677421305628),
 (4955, 0.49580639054625736),
 (4636, 0.4133953463507144),
 (4644, 0.4108780000140545),
 (3587, 0.3916564199419312)]

In [14]:
with open(
    r"E:\Reel-Recommendation-Engine\models\svd_model.pkl",
    "wb"
) as f:

    pickle.dump(
        svd_model,
        f
    )

print("SVD Model Saved")

SVD Model Saved


In [15]:
with open(
    r"E:\Reel-Recommendation-Engine\models\all_videos.pkl",
    "wb"
) as f:

    pickle.dump(
        all_videos,
        f
    )

print("Video List Saved")

Video List Saved


In [16]:
svd_df["recommendation_score"].value_counts().head(20)

recommendation_score
0.006580    21
0.222692    18
0.058562    18
0.662518    14
0.998675    13
0.025287    13
0.279348    13
0.002734    12
0.016672    12
0.027953    11
0.187247    11
0.031525    11
0.044409    11
0.171168    10
0.104374    10
0.008392    10
0.008967     9
0.041977     9
0.255117     8
0.188101     8
Name: count, dtype: int64

In [17]:
print(
    "Unique Scores:",
    svd_df["recommendation_score"].nunique()
)

print(
    "Zero Scores:",
    (svd_df["recommendation_score"] == 0).sum()
)

print(
    "One Scores:",
    (svd_df["recommendation_score"] == 1).sum()
)

Unique Scores: 1402471
Zero Scores: 1
One Scores: 1


In [18]:
reader = Reader(
    rating_scale=(0,1)
)

In [19]:
prediction = svd_model.predict(
    sample_user,
    sample_video
)

print(prediction.est)

0.06867298655409385


In [20]:
svd_df["recommendation_score"].isnull().sum()

0

In [21]:
print(svd_df["recommendation_score"].min())
print(svd_df["recommendation_score"].max())

0.0
1.0


In [22]:
print(len(trainset.all_users()))
print(len(trainset.all_items()))

26981
7536


In [23]:
predictions = svd_model.test(testset)

pred_df = pd.DataFrame(
    [
        (pred.uid,
         pred.iid,
         pred.r_ui,
         pred.est)
        for pred in predictions
    ],
    columns=[
        "user_id",
        "video_id",
        "actual",
        "predicted"
    ]
)

pred_df.head()

,user_id,video_id,actual,predicted
0,20691,1321,0.085211,0.097243
1,23162,6687,0.113560,0.111447
2,5038,6504,0.156586,0.153898
3,23742,5757,0.000642,0.000000
4,17501,6151,0.076453,0.077631


In [24]:
pred_df["predicted"].describe()

count    281930.000000
mean          0.094017
std           0.134324
min           0.000000
25%           0.019614
50%           0.056014
75%           0.107075
max           1.000000
Name: predicted, dtype: float64

In [25]:
pred_df = pd.DataFrame(
    [
        (
            pred.uid,
            pred.iid,
            pred.r_ui,
            pred.est
        )
        for pred in predictions
    ],
    columns=[
        "user_id",
        "video_id",
        "actual",
        "predicted"
    ]
)

pred_df["predicted"].describe()

count    281930.000000
mean          0.094017
std           0.134324
min           0.000000
25%           0.019614
50%           0.056014
75%           0.107075
max           1.000000
Name: predicted, dtype: float64

In [26]:
print(
    svd_df["recommendation_score"].value_counts().head(20)
)

recommendation_score
0.006580    21
0.222692    18
0.058562    18
0.662518    14
0.998675    13
0.025287    13
0.279348    13
0.002734    12
0.016672    12
0.027953    11
0.187247    11
0.031525    11
0.044409    11
0.171168    10
0.104374    10
0.008392    10
0.008967     9
0.041977     9
0.255117     8
0.188101     8
Name: count, dtype: int64
